# 02 – Spatial Join/Snap/Add to SWORD
**Purpose:** Join/ Snap and add different Dataset to the SWORD data.


**Workflow:**

2.0 | Imports

2.1 | Snap Points to Line

&nbsp;&nbsp;&nbsp;&nbsp;2.1.1 | Snap GDW points to SWORD lines

2.2 | Spatial Join

&nbsp;&nbsp;&nbsp;&nbsp;2.2.1 | Join GRITv1.0 strahler order information to SWORD

To-Do:
- [ ] erstellte outputs aufeinander aufbauen lassen, sodass der SWORD Datensatz immer erweitert wird.
- [ ] Muss noch im notebook den SWORD datensatzaufbau einbauen und übersichtlicher gestalten
- [ ] sections einfügen, um zum SWORD hinzugefügte Datensätze mit dem original zu vergleichen

---
## 2.0 | Imports

In [ ]:
from shapely.geometry import Point
import geopandas as gpd
import pandas as pd
import numpy as np
import os
import sys
import fiona
import matplotlib.pyplot as plt
import contextily as ctx
import webbrowser

sys.path.append(os.path.join("..", "src"))
from _0_config_paths import DATA_RAW, DATA_PROCESSED, DATA_OUTPUT
from _2_3_raster_join import extract_raster_values

#### Input / Output cell:

In [ ]:
SWORD = os.path.join(DATA_RAW, "0_data", "Naryn_example", "Naryn_SWORD_reaches.gpkg")

#--- Section 2.1.1 ------------------------------------------------------------
# Import GlobalDamWatch (GDW) points
GDW  = os.path.join(DATA_RAW, "0_data", "Naryn_example", "Naryn_GDW_barriers_v1_0.gpkg")
OUT_2_1_1 = os.path.join(DATA_PROCESSED, "snapped_Naryn_GDW_barriers_v1_0.gpkg")

#--- Section 2.2.1 ------------------------------------------------------------
# Inputs:
SWORD_CLIPPED_PATH = os.path.join(DATA_PROCESSED, "sword_naryn_clip.gpkg")
GRIT_PATH  = os.path.join(DATA_RAW, "0_data", "Naryn_example", "Naryn_GRITv1.0.gpkg")
GRIT_LAYER = None

# Outputs:
OUT_2_2_1 = os.path.join(DATA_PROCESSED, "sword_naryn_GRITv1.0_joined.gpkg")
OUT_MAP_2_2_1    = os.path.join(DATA_OUTPUT, "02_join_GRITv1.0_map.html")

#--- Section 2.3.1 ------------------------------------------------------------
# Load data STI:
# Each entry: (col_name, raster_path)
# For later: Adding all rasters which should be added the same way:
RASTER_JOINS = [
    ("STI_glowabio", os.path.join(DATA_RAW, "0_data", "Naryn_example", "glowabio", "Naryn_sti.tif")),
    # ("gee_JRC_...", os.path.join(DATA_RAW, "...", "gee_JRC_...")),
]

OUT_2_3_1 = os.path.join(DATA_PROCESSED, "sword_naryn_GRITv1.0_glowabio.gpkg")

# # ============================================================
# print("Input / Output paths set:")
# print("Section Section 2.1.1 GRIT Spatial Join:")
# print(f"IN  GRIT: {GDW}")
# print(f"OUT: {OUT_2_1_1}")


---
## 2.1 | Snap Points to Line
### 2.1.1 | Snap GDW points to SWORD lines

To-Do:
- [ ] Some of the original GDW points are located quite far from the feature, which is why the spatial deviation from the dam remains quite high even after snapping. 


<u>References:</u>

Ward, B. (2019): "How to leverage Geopandas for faster snapping of points to lines". URL: https://medium.com/@brendan_ward/how-to-leverage-geopandas-for-faster-snapping-of-points-to-lines-6113c94e59aa

Lehner, B. et al.(2024): Global Dam Watch database version 1.0. URL: https://figshare.com/articles/dataset/Global_Dam_Watch_database_version_1_0/25988293

https://www.globaldamwatch.org/database


In [ ]:
# See available layers in the GeoPackage
print(fiona.listlayers(GDW))
# transforming GDW to vector data
gdw = gpd.read_file(GDW)
print(gdw.geometry.geom_type.unique())

# Inspecting the data:
print(gdw.head())
print(gdw.columns)
print(gdw.info())
print(gdw.crs)

#gdw.columns = gdw.columns.str.lower()
print(gdw.columns)

In [ ]:
# Filtering Instream vs. Offstrem Features
# Checking fo individual Values in certain column:
gdw["INSTREAM"].unique()
gdw = gdw[gdw["INSTREAM"].str.lower() == "instream"]

gdw["INSTREAM"].unique()

In [ ]:
# Transform SWORD to vectorized data:
sword = gpd.read_file(SWORD)
print(sword.geometry.geom_type.unique())

# transform the data to a CRS in meters:
print(gdw.crs, sword.crs)
gdw = gdw.to_crs("EPSG:32643")
sword = sword.to_crs("EPSG:32643")
print(gdw.crs, sword.crs)

In [ ]:
# creating spatial index on sword to select only features which overlap with gdw bounding box:
sword.sindex

# create bounding boxes or gdw:
offset = 100
gdw_bbox = gdw.bounds + [-offset, -offset, offset, offset]

hits = gdw_bbox.apply(lambda row: list(sword.sindex.intersection(row)), axis=1)
print(hits)

# Check if one gdw point has no spatial matching line:
has_match = hits.apply(len) > 0
print("All gdw points have a sword line match: ", has_match.all())
# Check which point has no line:
gdw_no_match = gdw[~has_match]
print("No match:", gdw_no_match)

In [ ]:
# create a longer DataFrame with one row for each element in the list of line indexes:
tmp = pd.DataFrame({
    # index of points table
    "gdw_idx": np.repeat(hits.index, hits.apply(len)),
    # ordinal position of line - access via iloc later
    "sword_i": np.concatenate(hits.values)
})

# Join back to the sword on sword_i; we use reset_index() to give us the ordinal position of each line:
tmp = tmp.join(sword.reset_index(drop=True), on="sword_i")

# Join back to the original gdws to get their geometry rename the gdw geometry as "point":
tmp = tmp.join(gdw.geometry.rename("point"), on="gdw_idx")

# Convert back to a GeoDataFrame, so we can do spatial ops
tmp = gpd.GeoDataFrame(tmp, geometry="geometry", crs=gdw.crs)

Find the closest sword line to each gdw point:

In [ ]:
tmp["snap_dist"] = tmp.geometry.distance(gpd.GeoSeries(tmp.point))

# Discard any sword lines that are greater than tolerance from gdw pints:
# Print mean, median and biggest distance for all calculated distances:
# print("All calculated distances:\n",
#     "Mean:", tmp["snap_dist"].mean(),
#       "|Median:", tmp["snap_dist"].median(),
#       "|Max:", tmp["snap_dist"].max())
print(tmp["snap_dist"].describe())

# Identify GDW points that matched more than one SWORD line
match_counts = tmp.groupby("gdw_idx")["sword_i"].transform("count")

# Keep only rows where the GDW point has multiple SWORD matches
tmp_multiple = tmp[match_counts > 1]
# Print mean, median and biggest distance from all gdw points with multiple sword line matches:
print(tmp_multiple["snap_dist"].describe())

In [ ]:
# Discarding any sword lines that are greater than tolerance from gdw points:
tolerance = 164
tmp = tmp.loc[tmp.snap_dist <= tolerance]

# Sort on ascending snap distance, so that closest goes to top
tmp = tmp.sort_values(by=["snap_dist"])

# group by the index of the gdw points and take the first, which is the closest line 
closest = tmp.groupby("gdw_idx").first()
# construct a GeoDataFrame of the closest lines
closest = gpd.GeoDataFrame(closest, geometry="geometry")

Find the point on the line that is closest to the original point:

In [ ]:
# Position of nearest gdw point from start of the sword line
pos = closest.geometry.project(gpd.GeoSeries(closest.point))
# Get new point location geometry
snapped_pts = closest.geometry.interpolate(pos)

#Identify the columns we want to copy from the closest line to the point, such as a line ID.
print(closest.columns)
line_columns = ["sword_i", "reach_id", "network"]
# Create a new GeoDataFrame from the columns from the closest line and new point geometries (which will be called "geometries")
snapped = gpd.GeoDataFrame(
closest[line_columns], geometry = snapped_pts)

# Join back to the original points:
updated_points = gdw.drop(columns=["geometry"]).join(snapped)
# You may want to drop any that didn't snap, if so:
updated_points = updated_points.dropna(subset=["geometry"])

Export snapped GDW Points:

In [ ]:
# Export snapped gdw points as gpkg:
# Making sure gpd is a GeoDataFrame:
updated_points = gpd.GeoDataFrame(updated_points, geometry="geometry", crs=gdw.crs)

#output_path = os.path.join(DATA_PROCESSED, "snapped_gdw_points.gpkg")

updated_points.to_file(
    OUT_2_1_1,
    layer="snapped_Naryn_GDW_barriers_v1_0",
    driver="GPKG"
)

---
## 2.2 | Spatial Join
### 2.2.1 | Join GRITv1.0 strahler order information to SWORD

What I want to do:

I want to define a boundary for the egg that includes all the reaches within it. To do this, I want to use the strahler order, which I think works pretty well based on the GRITv1.0 dataset. This means that a single strahler order will contain multiple SWORD reaches. I then want to create one egg per strahler. 

To-Do:
- [ ] How should I handle reaches which falls within the coverage of multiple strahler? 
- [ ] Create src file for 2.2.1?


<u>References:</u>

Wortmann, M., Slater, L., Hawker, L., Liu, Y., Neal, J., Zhang, B., et al. (2025) URL: https://aquaknow.jrc.ec.europa.eu/dataset/global-river-topology-grit

Wortmann, M., Slater, L., Hawker, L., Liu, Y., Neal, J., Zhang, B., et al. (2025). Global River Topology (GRIT): A bifurcating river hydrography. Water Resources Research, 61, e2024WR038308. https://doi.org/10.1029/2024WR038308

In [ ]:
# ============================================================
# CONFIGURATION 

# Columns to take from GRIT:
GRIT_COLS_TO_JOIN = [
    "strahler_order",  # main information
    "global_id",       # provenance reference
    "geometry"         # required for spatial join
]

# How to Rename columns after join (Format: {"original_name": "new_name"}):
GRIT_RENAME = {
    "global_id": "global_id_GRITv1.0",
    "strahler_order": "strahler_order_GRITv1.0"
}

# tolerance for geometry mismatch between datasets:
MAX_DISTANCE_METERS = 100  

Load Data:

In [ ]:
# Load SWORD:
sword = gpd.read_file(SWORD_CLIPPED_PATH)
print(f"SWORD reaches loaded : {len(sword)}")
print(f"SWORD CRS: {sword.crs}")
print(f"GRIT columns: {sword.columns.tolist()}")

# Load GRIT
grit_layers = fiona.listlayers(GRIT_PATH)
grit_layer  = GRIT_LAYER if GRIT_LAYER else grit_layers[0]
grit_raw = gpd.read_file(GRIT_PATH, layer=grit_layer)

print(f"\nGRIT features loaded : {len(grit_raw)}")
print(f"GRIT CRS: {grit_raw.crs}")
print(f"GRIT columns: {grit_raw.columns.tolist()}")


Prepare GRIT

In [ ]:
# Only keeping the needed columns:
# Check that all requested columns actually exist in GRIT:
missing_cols = [c for c in GRIT_COLS_TO_JOIN if c != "geometry"
                and c not in grit_raw.columns]
if missing_cols:
    print(f"Columns not found in GRIT: {missing_cols}")
    print("Check column names in GRIT and update GRIT_COLS_TO_JOIN")
else:
    print("All requested columns found in GRIT")

# Selecting wanted column:
cols_to_keep = [c for c in GRIT_COLS_TO_JOIN if c in grit_raw.columns]
grit = grit_raw[cols_to_keep].copy()

print(f"\nGRIT prepared: {len(grit)} features, columns: {grit.columns.tolist()}")

In [ ]:
# transform the data to a CRS in meters:
print(grit.crs, sword.crs)
grit = grit.to_crs("EPSG:32643")
sword = sword.to_crs("EPSG:32643")
print(grit.crs, sword.crs)

SPATIAL JOIN

Creating points along the SWORD reaches and assigning each point to its nearest GRIT feature. By majority vote, the strahler order of GRIT is choosen for each reach.
The problem with sjoin_nearest was, that only compared thenearest point between two geometries which lead to missclassification.

In [ ]:
# sample a point every 50 meters along each reach:
SAMPLE_DISTANCE_M = 50

# For each reach, calculate how many sample points needed:
reach_lengths = sword.geometry.length
n_samples = (reach_lengths / SAMPLE_DISTANCE_M).astype(int).clip(lower=1)

# Build arrays of reach indices and distances vectorized
reach_indices = np.repeat(sword.index.values, n_samples.values)
distances     = np.concatenate([
    np.linspace(0, length, n)
    for length, n in zip(reach_lengths.values, n_samples.values)
])

# Interpolate points along each reach geometry
reach_geoms_repeated = sword.geometry.loc[reach_indices]
sample_points = reach_geoms_repeated.interpolate(distances)

# Build a GeoDataFrame of all sample points
samples = gpd.GeoDataFrame(
    {"sword_index": reach_indices},
    geometry=sample_points.values,
    crs=sword.crs
)

print(f"  SWORD reaches     : {len(sword)}")
print(f"  Sample points     : {len(samples)}")
print(f"  Avg points/reach  : {len(samples)/len(sword):.1f}")

Assign majority strahler order:

For each reach, count how many sample points were assigned to each strahler_order → the most frequent one wins. In case of a tie, the lower strahler order wins (more conservative).

In [ ]:
#Assign each sample point to nearest GRIT feature:
points_joined = gpd.sjoin_nearest(
    samples,
    grit[["strahler_order", "global_id", "geometry"]],
    how="left",
    max_distance=MAX_DISTANCE_METERS,
    #distance_col="dist_to_grit_m"
)

# Majority vote per SWORD reach:
def majority_vote(series):
    """Return the most frequent value. Ties go to the lower value."""
    return series.value_counts().idxmax()

# Group by sword reach index and take majority strahler_order
majority = (points_joined
    .dropna(subset=["strahler_order"])           # ignore unmatched points
    .groupby("sword_index")["strahler_order"]
    .agg(majority_vote)
    .rename("strahler_order_majority"))

# Also take the global_id of the majority-winning GRIT feature:
majority_id = (points_joined
    .dropna(subset=["strahler_order"])
    .groupby("sword_index")
    .apply(lambda df: df.loc[
        df["strahler_order"] == majority_vote(df["strahler_order"]),
        "global_id"
    ].iloc[0])
    .rename("global_id_majority"))

# Join majority result back to SWORD:
sword_joined = sword.join(majority).join(majority_id)

# Rename to final column names
sword_joined = sword_joined.rename(columns={
    "strahler_order_majority": "strahler_order_GRITv1.0",
    "global_id_majority"     : "global_id_GRITv1.0"
})

# Quality report
n_matched   = sword_joined["strahler_order_GRITv1.0"].notna().sum()
n_unmatched = sword_joined["strahler_order_GRITv1.0"].isna().sum()

print(f"\nRows total: {len(sword_joined)}")
print(f"Rows with match: {n_matched}")
print(f"Rows without match: {n_unmatched}")
print(f"\nReaches per strahler_order_GRITv1.0:")
print(sword_joined["strahler_order_GRITv1.0"].value_counts().sort_index())

#Add assignment confidence score:
confidence = (points_joined
    .dropna(subset=["strahler_order"])
    .groupby("sword_index")
    .apply(lambda df: (
        df["strahler_order"] == majority_vote(df["strahler_order"])
    ).mean())
    .rename("grit_majority_confidence"))

sword_joined = sword_joined.join(confidence)

print("\nConfidence score distribution:")
print(sword_joined["grit_majority_confidence"].describe())

Clean Table a little bit:

In [ ]:
# Drop the auto-generated index column from sjoin
sword_joined = sword_joined.drop(columns=["index_right"], errors="ignore")

# Rename columns
sword_joined = sword_joined.rename(columns=GRIT_RENAME)

print("Columns after join and rename:")
print(sword_joined.columns.tolist())

# Quick check: what do the new columns look like?
# Quick check: what do the new columns look like?
sword_joined[["reach_id", "strahler_order_GRITv1.0",
              "global_id_GRITv1.0", 
              "grit_majority_confidence"]].head(10)

In [ ]:
# Convert to integer:
INT_COLS = ["strahler_order_GRITv1.0", "global_id_GRITv1.0"]

for col in INT_COLS:
    if col in sword_joined.columns:
        sword_joined[col] = sword_joined[col].astype("Int64")
        print(f"Converted to Int64: {col}")
    else:
        print(f"Column not found, skipping: {col}")

# Verify
print("\nData types after conversion:")
print(sword_joined[INT_COLS].dtypes)
print("\nSample:")
print(sword_joined[["reach_id"] + INT_COLS].head())

Checking how clearly a SWORD reahc has been assigned to a strahler by grit_majority_confidence. "1" means that all points (100%) of a SWORD habe been assigned to one strahler.

In [ ]:
print("Majority confidence score [0–1]:")
print(sword_joined["grit_majority_confidence"].describe())

# Flag ambiguous reaches (confidence below 0.6)
ambiguous = sword_joined["grit_majority_confidence"] < 0.6
print(f"\nAmbiguous reaches (confidence < 0.6): {ambiguous.sum()}")

print("\nReaches per strahler_order_GRITv1.0:")
print(sword_joined["strahler_order_GRITv1.0"].value_counts().sort_index())

Export:

In [ ]:
os.makedirs(DATA_PROCESSED, exist_ok=True)
sword_joined.to_file(OUT_2_2_1, driver="GPKG")
print(f"Saved: {OUT_2_2_1}")

INTERACTIVE MAP – color reaches by strahler_order_GRITv1.0

In [ ]:
m = sword_joined.explore(
    column="strahler_order_GRITv1.0",
    cmap="RdYlBu",
    tooltip=["reach_id", "river_name", "strahler_order_GRITv1.0",
             "strm_order", "grit_majority_confidence"],
    tiles="CartoDB positron",
    legend=True
)

m.save(OUT_MAP_2_2_1)
webbrowser.open(OUT_MAP_2_2_1)
print(f"Map saved: {OUT_MAP_2_2_1}")

## 2.3 | Extracting/Sampling raster values to SWORD reaches

Since the reaches can be offset by up to 200 meters from the actual river, a buffer is set around the reaches,depending on the reach width (column: "width"), to extract pixels from raster data. 

To-Do:

- [ ] I don't like the buffer solution because, with the current logic, it could end up extracting an unnecessary number of incorrect pixels. Maybe I should use a flow direction grid to better specify the direction of the buffer?
- [ ] In "raster_join.py" add to function "def extract_raster_values" that only pixels which overlap the buffer with a min. area of.... =< 50%? are included in the mean/median calculation.

### 2.3.1 | Global freshwater biodiversity research: Stream Transportation Index (STI)


https://glowabio.org/

https://hydrography.org/hydrography90m/hydrography90m_layers/



In [ ]:
#Reload joined dataset
sword_joined = gpd.read_file(
    os.path.join(DATA_PROCESSED, "sword_naryn_GRITv1.0_joined.gpkg"))

print(f"Reloaded: {len(sword_joined)} reaches")
print(f"Columns: {sword_joined.columns.tolist()}")

width-based buffer extraction:

In [ ]:
sword_with_raster = sword_joined.copy()

for col_name, raster_path in RASTER_JOINS:
    print(f"{'='*50}")
    print(f"Processing : {col_name}")
    print(f"Raster     : {raster_path}")
    print(f"File exists: {os.path.exists(raster_path)}")
    sword_with_raster = extract_raster_values(
        sword_with_raster,
        raster_path,
        col_name,
        width_col="width"
    )

In [ ]:
# Inspect raster join results:
raster_cols = [c for name, _ in RASTER_JOINS
               for c in [f"{name}_mean", f"{name}_median"]]

print("Sample:")
sword_with_raster[["reach_id", "width"] + raster_cols].head(10)

print("\nStatistics:")
sword_with_raster[raster_cols].describe()

In [ ]:
# Export
os.makedirs(DATA_PROCESSED, exist_ok=True)
sword_with_raster.to_file(OUT_2_3_1, driver="GPKG")
print(f"Saved: {OUT_2_3_1}")